In [3]:
import re
import math
import pandas as pd
import numpy as np
from collections import Counter
import nltk
from nltk.corpus import stopwords
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# ==============================================================================
# 1. LEITURA E CRIAÇÃO DO RÓTULO
# ==============================================================================

df = pd.read_csv("/content/tfidf.csv", usecols=["titulo", "tags"])

def rotular(tags):
    if pd.isna(tags):
        return None
    t = tags.lower()
    plats = sum([
        "pc"      in t,
        "ps5"     in t or "ps4" in t or "ps6" in t,
        "xbox"    in t,
        "nintendo" in t,
        "android" in t or "ios" in t,
    ])
    if plats == 0:
        return None
    return "multiplataforma" if plats > 1 else "exclusivo"

df["label"] = df["tags"].apply(rotular)
df = df.dropna(subset=["label", "titulo"]).copy()

print(f"Total de notícias: {len(df)}")
print("Distribuição de classes:")
print(df["label"].value_counts())

# ==============================================================================
# 2. PRÉ-PROCESSAMENTO: LIMPEZA E REMOÇÃO DE STOPWORDS
# ==============================================================================

nltk.download('stopwords') # Added this line to download the stopwords corpus
stopwords_pt = set(stopwords.words("portuguese"))
stopwords_en = set(stopwords.words("english"))
stopwords_todos = stopwords_pt | stopwords_en

# palavras relacionadas a plataforma que dariam data leakage
leakage = {"pc", "ps5", "ps4", "ps6", "xbox", "nintendo", "android", "ios",
           "switch", "playstation", "microsoft", "sony"}

stopwords_finais = stopwords_todos | leakage

def limpar(texto):
    texto = texto.lower()
    texto = re.sub(r"[^\w\sáéíóúãõâêîôûàèìòùç]", " ", texto)
    tokens = texto.split()
    tokens = [t for t in tokens if t not in stopwords_finais and len(t) > 2]
    return tokens

df["tokens"] = df["titulo"].apply(limpar)

# exemplo do pré-processamento
print("\n--- Exemplo de pré-processamento ---")
for _, row in df.head(3).iterrows():
    print(f"  Original : {row['titulo']}")
    print(f"  Tokens   : {row['tokens']}")
    print()

# ==============================================================================
# 3. TF-IDF DO ZERO
# ==============================================================================

def calcular_tfidf(corpus_tokens):
    N = len(corpus_tokens)

    # TF: frequência relativa por documento
    tf_docs = []
    for tokens in corpus_tokens:
        contagem = Counter(tokens)
        total = len(tokens) if tokens else 1
        tf_docs.append({t: c / total for t, c in contagem.items()})

    # DF: em quantos documentos cada termo aparece
    df_terms = Counter()
    for tokens in corpus_tokens:
        df_terms.update(set(tokens))

    # IDF: log(N / df+1) + 1  (suavizado para evitar divisão por zero)
    idf = {t: math.log(N / (df_terms[t] + 1)) + 1 for t in df_terms}

    # Vocabulário ordenado
    vocab = sorted(idf.keys())
    vocab_idx = {t: i for i, t in enumerate(vocab)}

    # Matriz TF-IDF
    matriz = np.zeros((N, len(vocab)), dtype=np.float32)
    for i, tf in enumerate(tf_docs):
        for termo, tf_val in tf.items():
            if termo in vocab_idx:
                j = vocab_idx[termo]
                matriz[i, j] = tf_val * idf[termo]

    # Normalização L2 por linha
    normas = np.linalg.norm(matriz, axis=1, keepdims=True)
    normas[normas == 0] = 1
    matriz = matriz / normas

    return matriz, vocab, idf

print("Calculando TF-IDF do zero...")
X, vocab, idf = calcular_tfidf(df["tokens"].tolist())
y = (df["label"] == "multiplataforma").astype(int).values

print(f"Dimensões da matriz TF-IDF: {X.shape}")
print(f"Vocabulário: {len(vocab)} termos únicos")

# top termos por IDF (mais informativos = IDF alto)
print("\n--- Top 15 termos mais informativos (IDF alto) ---")
top_idf = sorted(idf.items(), key=lambda x: -x[1])[:15]
for termo, val in top_idf:
    print(f"  {termo:<20} IDF = {val:.3f}")

# ==============================================================================
# 4. TREINO / TESTE
# ==============================================================================

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTreino: {X_treino.shape[0]} | Teste: {X_teste.shape[0]}")

# ==============================================================================
# 5. MODELOS
# ==============================================================================

modelos = {
    "Regressão Logística": LogisticRegression(max_iter=1000, C=1.0),
    "Naive Bayes"        : MultinomialNB(alpha=0.5),
    "Random Forest"      : RandomForestClassifier(n_estimators=200, random_state=42),
}

resultados = {}

print("\n--- Avaliação dos modelos ---")
for nome, modelo in modelos.items():
    modelo.fit(X_treino, y_treino)
    y_prob = modelo.predict_proba(X_teste)[:, 1]
    y_pred = modelo.predict(X_teste)

    auc = roc_auc_score(y_teste, y_prob)
    cv  = cross_val_score(modelo, X, y, cv=5, scoring="roc_auc").mean()

    resultados[nome] = {"modelo": modelo, "auc_teste": auc, "auc_cv": cv}

    print(f"\n{nome}")
    print(f"  AUC teste  : {auc:.4f}")
    print(f"  AUC CV (5) : {cv:.4f}")
    print(classification_report(y_teste, y_pred,
                                target_names=["exclusivo", "multiplataforma"],
                                digits=3))

# ==============================================================================
# 6. ANÁLISE DO MELHOR MODELO
# ==============================================================================

melhor_nome = max(resultados, key=lambda k: resultados[k]["auc_teste"])
melhor      = resultados[melhor_nome]["modelo"]

print(f"=== Melhor modelo: {melhor_nome} ===")

# Termos mais associados a cada classe (Regressão Logística)
if hasattr(melhor, "coef_"):
    coefs = melhor.coef_[0]
    top_multi = [vocab[i] for i in np.argsort(-coefs)[:10]]
    top_excl  = [vocab[i] for i in np.argsort(coefs)[:10]]
    print("\nTermos mais associados a MULTIPLATAFORMA:")
    print(" ", top_multi)
    print("Termos mais associados a EXCLUSIVO:")
    print(" ", top_excl)

# ==============================================================================
# 7. TESTE COM FRASES NOVAS
# ==============================================================================

def prever(titulo, modelo, vocab_idx, idf):
    tokens = limpar(titulo)
    if not tokens:
        return None, None
    N_fake = 1
    tf = Counter(tokens)
    total = len(tokens)
    vetor = np.zeros((1, len(vocab_idx)), dtype=np.float32)
    for t, c in tf.items():
        if t in vocab_idx:
            tf_val = c / total
            vetor[0, vocab_idx[t]] = tf_val * idf.get(t, 0)
    norma = np.linalg.norm(vetor)
    if norma > 0:
        vetor /= norma
    prob = modelo.predict_proba(vetor)[0, 1]
    classe = "multiplataforma" if prob > 0.5 else "exclusivo"
    return classe, prob

vocab_idx = {t: i for i, t in enumerate(vocab)}

print("\n--- Previsões em títulos novos ---")
titulos_teste = [
    "Novo RPG lançado para todas as plataformas em simultâneo",
    "Sequela aguardada chega em exclusivo à consola japonesa",
    "Estúdio indie anuncia jogo de sobrevivência no Steam",
    "Remake clássico anunciado para consola da Sony",
    "Jogo de luta chega a todas as plataformas no verão",
]

for t in titulos_teste:
    classe, prob = prever(t, melhor, vocab_idx, idf)
    print(f"  [{classe:>15}  p={prob:.2f}]  {t}")

print("\n=== CONCLUÍDO ===")

Total de notícias: 1018
Distribuição de classes:
label
multiplataforma    627
exclusivo          391
Name: count, dtype: int64

--- Exemplo de pré-processamento ---
  Original : Tecmo Bowl será adaptado para filme
  Tokens   : ['tecmo', 'bowl', 'adaptado', 'filme']

  Original : Fãs adoram Super Mario Galaxy: O Filme
  Tokens   : ['fãs', 'adoram', 'super', 'mario', 'galaxy', 'filme']

  Original : Pokémon FireRed & LeafGreen inclui conteúdo ausente da versão original
  Tokens   : ['pokémon', 'firered', 'leafgreen', 'inclui', 'conteúdo', 'ausente', 'versão', 'original']

Calculando TF-IDF do zero...


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Dimensões da matriz TF-IDF: (1018, 2344)
Vocabulário: 2344 termos únicos

--- Top 15 termos mais informativos (IDF alto) ---
  tecmo                IDF = 7.232
  adaptado             IDF = 7.232
  ausente              IDF = 7.232
  transações           IDF = 7.232
  micro                IDF = 7.232
  detesta              IDF = 7.232
  pois                 IDF = 7.232
  tão                  IDF = 7.232
  mentiras             IDF = 7.232
  insomniac            IDF = 7.232
  moorcroft            IDF = 7.232
  project              IDF = 7.232
  existem              IDF = 7.232
  lembrar              IDF = 7.232
  drm                  IDF = 7.232

Treino: 814 | Teste: 204

--- Avaliação dos modelos ---

Regressão Logística
  AUC teste  : 0.8548
  AUC CV (5) : 0.8308
                 precision    recall  f1-score   support

      exclusivo      0.839     0.333     0.477        78
multiplataforma      0.699     0.960     0.809       126

       accuracy                          0.721       20

In [4]:
import re
import pandas as pd
import numpy as np
from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
from sklearn.preprocessing import MaxAbsScaler
import scipy.sparse as sp
import nltk

# ==============================================================================
# 1. LEITURA E ROTULAÇÃO
# ==============================================================================

df = pd.read_csv("/content/tfidf.csv", usecols=["titulo", "tags", "data"])

PLATS_LEAKAGE = {"pc", "ps5", "ps4", "ps6", "xbox", "nintendo", "android",
                 "ios", "switch", "playstation", "microsoft", "sony"}

TAGS_PLAT = {"pc", "ps5", "ps4", "ps6", "xbox series x", "xbox series s",
             "xbox one", "nintendo switch", "nintendo switch 2", "android",
             "ios", "playstation 4", "playstation 5", "playstation 6"}

def rotular(tags):
    if pd.isna(tags):
        return None
    t = tags.lower()
    plats = sum([
        "pc"       in t,
        "ps5"      in t or "ps4" in t or "ps6" in t,
        "xbox"     in t,
        "nintendo" in t,
        "android"  in t or "ios" in t,
    ])
    if plats == 0:
        return None
    return "multiplataforma" if plats > 1 else "exclusivo"

def tags_sem_plataforma(tags):
    if pd.isna(tags):
        return ""
    partes = [p.strip().lower() for p in tags.split(",")]
    return " ".join(p.replace(" ", "_") for p in partes if p not in TAGS_PLAT)

df["label"]        = df["tags"].apply(rotular)
df["tags_extra"]   = df["tags"].apply(tags_sem_plataforma)
df["data"]         = pd.to_datetime(df["data"], errors="coerce", utc=True)
df["mes"]          = df["data"].dt.month.fillna(0).astype(int)
df["ano"]          = df["data"].dt.year.fillna(0).astype(int)

df = df.dropna(subset=["label", "titulo"]).copy()

print(f"Total: {len(df)} notícias")
print(df["label"].value_counts())

# ==============================================================================
# 2. PRÉ-PROCESSAMENTO COM STEMMING
# ==============================================================================

nltk.download('stopwords') # Adicionando a linha para baixar as stopwords
nltk.download('rslp') # Adicionando a linha para baixar o stemmer rslp
sw_pt   = set(stopwords.words("portuguese"))
sw_en   = set(stopwords.words("english"))
# stopwords específicas do domínio de games
sw_dom  = {"jogo", "jogos", "novo", "nova", "novos", "novas", "trailer",
           "gameplay", "vídeo", "video", "review", "análise", "anuncio",
           "anúncio", "lançamento", "chegada", "chega", "chegou",
           "disponível", "disponivel", "game", "games"}
SW_FINAL = sw_pt | sw_en | sw_dom | PLATS_LEAKAGE

stemmer = RSLPStemmer()

def limpar(texto, usar_stem=True):
    texto = texto.lower()
    texto = re.sub(r"[^\w\sáéíóúãõâêîôûàèìòùç]", " ", texto)
    tokens = [t for t in texto.split() if t not in SW_FINAL and len(t) > 2]
    if usar_stem:
        tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)

df["texto_limpo"] = df["titulo"].apply(limpar)

# ==============================================================================
# 3. FEATURES TF-IDF (sklearn) COM UNIGRAMAS + BIGRAMAS
# ==============================================================================

tfidf_titulo = TfidfVectorizer(
    ngram_range = (1, 2),
    min_df      = 2,
    max_df      = 0.95,
    sublinear_tf = True,
)

tfidf_tags = TfidfVectorizer(
    ngram_range  = (1, 1),
    min_df       = 2,
    max_df       = 0.95,
    token_pattern = r"[^\s]+",
)

# ==============================================================================
# 4. TREINO / TESTE
# ==============================================================================

y = (df["label"] == "multiplataforma").astype(int).values

idx_tr, idx_te = train_test_split(
    df.index, test_size=0.2, random_state=42,
    stratify=y
)

df_tr = df.loc[idx_tr]
df_te = df.loc[idx_te]
y_tr  = y[df.index.get_indexer(idx_tr)]
y_te  = y[df.index.get_indexer(idx_te)]

X_titulo_tr = tfidf_titulo.fit_transform(df_tr["texto_limpo"])
X_titulo_te = tfidf_titulo.transform(df_te["texto_limpo"])

X_tags_tr   = tfidf_tags.fit_transform(df_tr["tags_extra"])
X_tags_te   = tfidf_tags.transform(df_te["tags_extra"])

# features temporais
def feats_tempo(sub):
    mes  = sub["mes"].values.reshape(-1, 1) / 12.0
    ano  = ((sub["ano"] - 2020) / 5.0).clip(0, 1).values.reshape(-1, 1)
    return sp.hstack([sp.csr_matrix(mes), sp.csr_matrix(ano)])

X_tempo_tr = feats_tempo(df_tr)
X_tempo_te = feats_tempo(df_te)

X_tr = sp.hstack([X_titulo_tr, X_tags_tr, X_tempo_tr])
X_te = sp.hstack([X_titulo_te, X_tags_te, X_tempo_te])

print(f"\nDimensão treino : {X_tr.shape}")
print(f"Dimensão teste  : {X_te.shape}")

# ==============================================================================
# 5. GRID SEARCH NO NAIVE BAYES
# ==============================================================================

print("\n--- Grid search: alpha do Naive Bayes ---")

gs = GridSearchCV(
    MultinomialNB(),
    param_grid  = {"alpha": [0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0]},
    cv          = 5,
    scoring     = "roc_auc",
    n_jobs      = -1,
)
gs.fit(X_tr, y_tr)

melhor_alpha = gs.best_params_["alpha"]
print(f"Melhor alpha: {melhor_alpha}  |  AUC CV: {gs.best_score_:.4f}")

# ==============================================================================
# 6. MODELOS FINAIS
# ==============================================================================

nb_base   = MultinomialNB(alpha=melhor_alpha)
nb_cal    = CalibratedClassifierCV(MultinomialNB(alpha=melhor_alpha), cv=5)
rl        = LogisticRegression(max_iter=1000, C=1.0)

modelos = {
    "Naive Bayes (tunado)"     : nb_base,
    "Naive Bayes (calibrado)"  : nb_cal,
    "Regressão Logística"      : rl,
}

def calcular_auc(y, prob):
    pos = prob[y == 1]; neg = prob[y == 0]
    return (np.sum(np.subtract.outer(pos, neg) > 0) +
            0.5 * np.sum(np.subtract.outer(pos, neg) == 0)) / (len(pos) * len(neg))

print("\n--- Comparação de modelos ---")
resultados = {}
for nome, m in modelos.items():
    m.fit(X_tr, y_tr)
    prob = m.predict_proba(X_te)[:, 1]
    pred = m.predict(X_te)
    auc  = roc_auc_score(y_te, prob)
    cv   = cross_val_score(m, X_tr, y_tr, cv=5, scoring="roc_auc").mean()
    resultados[nome] = {"modelo": m, "prob": prob, "auc": auc, "cv": cv}
    print(f"\n{nome}")
    print(f"  AUC teste: {auc:.4f}  |  AUC CV: {cv:.4f}")
    print(classification_report(y_te, pred,
                                target_names=["exclusivo", "multiplataforma"],
                                digits=3))

# ==============================================================================
# 7. ENSEMBLE: MÉDIA DAS PROBABILIDADES (NB + RL)
# ==============================================================================

print("\n--- Ensemble: NB calibrado + Regressão Logística ---")

prob_ens = (resultados["Naive Bayes (calibrado)"]["prob"] +
            resultados["Regressão Logística"]["prob"]) / 2

auc_ens = roc_auc_score(y_te, prob_ens)
pred_ens = (prob_ens >= 0.5).astype(int)
cv_ens   = (resultados["Naive Bayes (calibrado)"]["cv"] +
            resultados["Regressão Logística"]["cv"]) / 2

print(f"  AUC ensemble: {auc_ens:.4f}  |  AUC CV (média): {cv_ens:.4f}")
print(classification_report(y_te, pred_ens,
                             target_names=["exclusivo", "multiplataforma"],
                             digits=3))

# ==============================================================================
# 8. THRESHOLD ÓTIMO (curva ROC)
# ==============================================================================

print("\n--- Threshold ótimo (ensemble) ---")

fpr, tpr, thresholds = roc_curve(y_te, prob_ens)
distancia = np.sqrt(fpr**2 + (1 - tpr)**2)
idx_opt   = np.argmin(distancia)
thr_opt   = thresholds[idx_opt]

pred_opt  = (prob_ens >= thr_opt).astype(int)
print(f"Threshold padrão (0.50):")
print(classification_report(y_te, pred_ens,
                             target_names=["exclusivo", "multiplataforma"],
                             digits=3))
print(f"Threshold ótimo  ({thr_opt:.2f}):")
print(classification_report(y_te, pred_opt,
                             target_names=["exclusivo", "multiplataforma"],
                             digits=3))

# ==============================================================================
# 9. ANÁLISE DE ERROS
# ==============================================================================

print("\n--- Análise de erros (ensemble, threshold ótimo) ---")

df_te_view = df.loc[idx_te].copy()
df_te_view["prob_multi"] = prob_ens
df_te_view["pred"]       = pred_opt
df_te_view["real"]       = y_te
df_te_view["erro"]       = df_te_view["pred"] != df_te_view["real"]
df_te_view["conf"]       = np.abs(prob_ens - 0.5)   # confiança do erro

erros = df_te_view[df_te_view["erro"]].nlargest(15, "conf")

print("Top 15 erros mais confiantes (modelo errou com certeza):")
for _, r in erros.iterrows():
    real_str = "multi" if r["real"] == 1 else "excl "
    pred_str = "multi" if r["pred"] == 1 else "excl "
    print(f"  real={real_str} pred={pred_str} p={r['prob_multi']:.2f}  {r['titulo'][:70]}")

# ==============================================================================
# 10. RESUMO FINAL
# ==============================================================================

print("\n" + "="*55)
print("RESUMO DAS MELHORIAS")
print("="*55)
print(f"{'Modelo':<35} {'AUC':>6}")
print("-"*43)
for nome, r in resultados.items():
    print(f"  {nome:<33} {r['auc']:.4f}")
print(f"  {'Ensemble NB + RL':<33} {auc_ens:.4f}")
print("="*55)

Total: 1018 notícias
label
multiplataforma    627
exclusivo          391
Name: count, dtype: int64


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Unzipping stemmers/rslp.zip.



Dimensão treino : (814, 1326)
Dimensão teste  : (204, 1326)

--- Grid search: alpha do Naive Bayes ---
Melhor alpha: 2.0  |  AUC CV: 0.9355

--- Comparação de modelos ---

Naive Bayes (tunado)
  AUC teste: 0.9447  |  AUC CV: 0.9355
                 precision    recall  f1-score   support

      exclusivo      0.941     0.615     0.744        78
multiplataforma      0.804     0.976     0.882       126

       accuracy                          0.838       204
      macro avg      0.873     0.796     0.813       204
   weighted avg      0.856     0.838     0.829       204


Naive Bayes (calibrado)
  AUC teste: 0.9438  |  AUC CV: 0.9361
                 precision    recall  f1-score   support

      exclusivo      0.909     0.769     0.833        78
multiplataforma      0.870     0.952     0.909       126

       accuracy                          0.882       204
      macro avg      0.889     0.861     0.871       204
   weighted avg      0.885     0.882     0.880       204


Regressão Lo